In [8]:
import PyPDF2
import fitz  # PyMuPDF

def analyze_pdf_content(pdf_path):
    """
    Analyze a PDF to check for text content and images
    """
    print(f"Analyzing PDF: {pdf_path}")
    print("-" * 50)
    
    # Check for text using PyPDF2
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        num_pages = len(reader.pages)
        print(f"Number of pages: {num_pages}")
        
        print("\nChecking for text content:")
        for page_num in range(num_pages):
            page = reader.pages[page_num]
            text = page.extract_text()
            if text.strip():
                print(f"Page {page_num + 1} contains text:")
                print(text[:200] + "..." if len(text) > 200 else text)
            else:
                print(f"Page {page_num + 1} has no extractable text")
    
    # Check for images using PyMuPDF
    doc = fitz.open(pdf_path)
    print("\nChecking for images:")
    for page_num in range(doc.page_count):
        page = doc[page_num]
        image_list = page.get_images()
        if image_list:
            print(f"Page {page_num + 1} contains {len(image_list)} images")
            # Print some details about each image
            for img_index, img in enumerate(image_list):
                xref = img[0]
                base_image = doc.extract_image(xref)
                if base_image:
                    print(f"  Image {img_index + 1}:")
                    print(f"    Size: {base_image.get('width')}x{base_image.get('height')}")
                    print(f"    Format: {base_image.get('ext')}")
        else:
            print(f"Page {page_num + 1} has no images")
    doc.close()

if __name__ == "__main__":
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Takeoff Drawing Example 2.pdf"
    try:
        analyze_pdf_content(pdf_path)
    except Exception as e:
        print(f"Error analyzing PDF: {e}")

Analyzing PDF: /Users/rc/Desktop/pdf/pdf-examples/Takeoff Drawing Example 2.pdf
--------------------------------------------------
Number of pages: 3

Checking for text content:
Page 1 has no extractable text
Page 2 has no extractable text
Page 3 has no extractable text

Checking for images:
Page 1 contains 1 images
  Image 1:
    Size: 2200x1700
    Format: png
Page 2 contains 1 images
  Image 1:
    Size: 2198x1700
    Format: png
Page 3 contains 1 images
  Image 1:
    Size: 2196x1700
    Format: png


In [11]:
%pip install opencv-python

  Using cached opencv_python-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl.metadata (20 kB)
Using cached opencv_python-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl (37.3 MB)

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import re
import cv2
import numpy as np

def preprocess_image(image):
    """
    Preprocess the image to improve OCR accuracy
    """
    # Convert PIL image to OpenCV format
    opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
    # Convert to grayscale
    gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
    
    # Apply thresholding to get black and white image
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Return to PIL format
    return Image.fromarray(thresh)

def extract_text_from_pdf(pdf_path):
    """
    Extract text from images in PDF using OCR
    """
    # Open the PDF
    doc = fitz.open(pdf_path)
    all_text = []
    
    for page_num in range(len(doc)):
        print(f"\nProcessing page {page_num + 1}...")
        page = doc[page_num]
        
        # Get image list
        image_list = page.get_images()
        
        for img_index, img in enumerate(image_list):
            # Get the image reference
            xref = img[0]
            
            # Extract image
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            image = Image.open(io.BytesIO(image_bytes))
            
            # Preprocess image
            processed_image = preprocess_image(image)
            
            # Perform OCR with custom configuration
            custom_config = r'--oem 3 --psm 6'  # Assume uniform text block
            text = pytesseract.image_to_string(processed_image, config=custom_config)
            
            # Store the text
            all_text.append(text)
            
            # Print raw text for debugging
            print(f"\nRaw extracted text from page {page_num + 1}:")
            print(text)
            
            # Try to find structured data
            lines = text.split('\n')
            for line in lines:
                if line.strip():
                    # Look for patterns of numbers followed by units
                    pattern = r'(\d+)?\s*(\d+(?:\.\d+)?)\s*([A-Za-z]+)\s*(.*)'
                    match = re.search(pattern, line)
                    if match:
                        no, qty, unit, desc = match.groups()
                        print("\nFound structured data:")
                        print(f"No: {no if no else 'N/A'}")
                        print(f"Quantity: {qty}")
                        print(f"Unit: {unit}")
                        print(f"Description: {desc.strip()}")
                        print("-" * 40)
    
    doc.close()
    return all_text

def main():
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Takeoff Drawing Example 2.pdf"
    try:
        extract_text_from_pdf(pdf_path)
    except Exception as e:
        print(f"Error processing PDF: {e}")

if __name__ == "__main__":
    main()


Processing page 1...

Raw extracted text from page 1:
i
_N EG) cc
a fief a [+ [racammas ae |
wat Pre pe [or [asiemna mac eam |
“a aa pp Lo | oe eens aan |
~ Pp ae fexemmaname |
, “ [oa fecoer namin
owe’ A [|
ap. el
ws ve
On GE ee om
x .
. 18°-150g RF
S "Sp CY BouT-UP |
eS CUTBACK TO 30 # oa <1
CUTBACK TO 30" ee Ci 2
Le Say & eee oe
18-1509 RF “
ain Sp ISSUED FOR
ao SRS
- 7-20-23
NOTES: fA 7.
1. SHOP TO HYDROTEST SPOOLS. E M b ]
2. PRIOR TO DEMO, FIELD TO USE TEMPORARY | on Ol mf Lt es
CRIBBING FOR SUPPORT OF AOV2E0 AT NOZZLE. ; Sov Aero 9 ark oc
10] REVISIONS Toe : OMCC/ETHYL PLANT PRO INS: EDGAR. SOTO PESONE: CASEY PRIDGEN .
ee PO31~23 BEAUMONT REFINERY POS IN_ ACCORDANCE WITH GP 03-18-01 UNLESS NOTED REL N/A
PT, PEPSI) REPLACE UNLEADED GASOLINE PIPING — [YP norwa Wecouee PY
8 | Ssvep FoR ConsTRUCTION ES [7aoas forme ae | TO_INSTALL FIGURE 8 BLIND AT TK 1212 [WBS cis ERM yer aE N/a
—— ‘OFS. PRESS. 285 PSIG [ pun FLOW UNE NUMBER [DWC NO. REV. No.
A | ISSUED FOR APPROVAL ES {6-19-23}ne

In [13]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import cv2
import numpy as np

def preprocess_image(image):
    """
    Preprocess the image to improve OCR accuracy
    """
    # Convert PIL image to OpenCV format
    opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
    # Convert to grayscale
    gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
    
    # Apply thresholding to get black and white image
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Return to PIL format
    return Image.fromarray(thresh)

def extract_and_display_all_text(pdf_path):
    """
    Extract and display ALL text from the PDF images
    """
    # Open the PDF
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        print(f"\n{'='*40}")
        print(f"PAGE {page_num + 1}")
        print(f"{'='*40}")
        
        page = doc[page_num]
        image_list = page.get_images()
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            image = Image.open(io.BytesIO(image_bytes))
            
            # Preprocess image
            processed_image = preprocess_image(image)
            
            # Try different PSM modes for better results
            psm_modes = [6, 3]  # 6 for uniform text block, 3 for fully automatic
            
            for psm_mode in psm_modes:
                custom_config = f'--oem 3 --psm {psm_mode}'
                text = pytesseract.image_to_string(processed_image, config=custom_config)
                
                # Split into lines and remove empty lines
                lines = [line.strip() for line in text.split('\n') if line.strip()]
                
                print(f"\nExtracted text (PSM mode {psm_mode}):")
                print("-" * 40)
                for line in lines:
                    print(line)
    
    doc.close()

def main():
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Takeoff Drawing Example 2.pdf"
    try:
        extract_and_display_all_text(pdf_path)
    except Exception as e:
        print(f"Error processing PDF: {e}")

if __name__ == "__main__":
    main()


PAGE 1

Extracted text (PSM mode 6):
----------------------------------------
i
_N EG) cc
a fief a [+ [racammas ae |
wat Pre pe [or [asiemna mac eam |
“a aa pp Lo | oe eens aan |
~ Pp ae fexemmaname |
, “ [oa fecoer namin
owe’ A [|
ap. el
ws ve
On GE ee om
x .
. 18°-150g RF
S "Sp CY BouT-UP |
eS CUTBACK TO 30 # oa <1
CUTBACK TO 30" ee Ci 2
Le Say & eee oe
18-1509 RF “
ain Sp ISSUED FOR
ao SRS
- 7-20-23
NOTES: fA 7.
1. SHOP TO HYDROTEST SPOOLS. E M b ]
2. PRIOR TO DEMO, FIELD TO USE TEMPORARY | on Ol mf Lt es
CRIBBING FOR SUPPORT OF AOV2E0 AT NOZZLE. ; Sov Aero 9 ark oc
10] REVISIONS Toe : OMCC/ETHYL PLANT PRO INS: EDGAR. SOTO PESONE: CASEY PRIDGEN .
ee PO31~23 BEAUMONT REFINERY POS IN_ ACCORDANCE WITH GP 03-18-01 UNLESS NOTED REL N/A
PT, PEPSI) REPLACE UNLEADED GASOLINE PIPING — [YP norwa Wecouee PY
8 | Ssvep FoR ConsTRUCTION ES [7aoas forme ae | TO_INSTALL FIGURE 8 BLIND AT TK 1212 [WBS cis ERM yer aE N/a
—— ‘OFS. PRESS. 285 PSIG [ pun FLOW UNE NUMBER [DWC NO. REV. No.
A | ISSUED FOR

In [14]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import cv2
import numpy as np
import re

def preprocess_image(image):
    """
    Preprocess the image to improve OCR accuracy
    """
    opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(thresh)

def extract_structured_data(text):
    """
    Extract no, quantity, unit, and description from text
    """
    items = []
    
    # Split into lines and process each line
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    for line in lines:
        # Pattern to match:
        # - Optional number at start (no)
        # - Number (can be decimal) for quantity
        # - Word for unit
        # - Rest of line for description
        pattern = r'(?P<no>\d+)?\s*(?P<qty>\d+(?:\.\d+)?)\s*(?P<unit>[A-Za-z]+)\s*(?P<description>.*)'
        match = re.search(pattern, line)
        
        if match:
            item = {
                'no': match.group('no') if match.group('no') else '',
                'qty': match.group('qty'),
                'unit': match.group('unit'),
                'description': match.group('description').strip()
            }
            items.append(item)
    
    return items

def process_pdf(pdf_path):
    """
    Process PDF and extract structured information
    """
    doc = fitz.open(pdf_path)
    all_items = []
    
    for page_num in range(len(doc)):
        print(f"\nProcessing page {page_num + 1}...")
        page = doc[page_num]
        image_list = page.get_images()
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))
            
            # Preprocess and OCR
            processed_image = preprocess_image(image)
            custom_config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(processed_image, config=custom_config)
            
            # Extract structured data
            items = extract_structured_data(text)
            all_items.extend(items)
    
    doc.close()
    return all_items

def display_results(items):
    """
    Display the filtered results in a clean format
    """
    print("\nExtracted Items:")
    print("=" * 60)
    
    for i, item in enumerate(items, 1):
        print(f"Item {i}:")
        print(f"  No: {item['no'] if item['no'] else 'N/A'}")
        print(f"  Quantity: {item['qty']}")
        print(f"  Unit: {item['unit']}")
        print(f"  Description: {item['description']}")
        print("-" * 60)

def main():
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Takeoff Drawing Example 2.pdf"
    try:
        # Process PDF and get structured data
        items = process_pdf(pdf_path)
        
        # Display results
        display_results(items)
        
    except Exception as e:
        print(f"Error processing PDF: {e}")

if __name__ == "__main__":
    main()


Processing page 1...

Processing page 2...

Processing page 3...

Extracted Items:
Item 1:
  No: 15
  Quantity: 0
  Unit: g
  Description: RF
------------------------------------------------------------
Item 2:
  No: 150
  Quantity: 9
  Unit: RF
  Description: “
------------------------------------------------------------
Item 3:
  No: N/A
  Quantity: 2
  Unit: E
  Description: 0 AT NOZZLE. ; Sov Aero 9 ark oc
------------------------------------------------------------
Item 4:
  No: 2
  Quantity: 3
  Unit: BEAUMONT
  Description: REFINERY POS IN_ ACCORDANCE WITH GP 03-18-01 UNLESS NOTED REL N/A
------------------------------------------------------------
Item 5:
  No: N/A
  Quantity: 7
  Unit: aoas
  Description: forme ae | TO_INSTALL FIGURE 8 BLIND AT TK 1212 [WBS cis ERM yer aE N/a
------------------------------------------------------------
Item 6:
  No: N/A
  Quantity: 285
  Unit: PSIG
  Description: [ pun FLOW UNE NUMBER [DWC NO. REV. No.
----------------------------------------

In [16]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import cv2
import numpy as np
import re

def preprocess_image(image):
    """
    Preprocess the image to improve OCR accuracy
    """
    opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(thresh)

def extract_bom_data(text):
    """
    Extract Bill of Materials data with specific columns:
    ID, QTY, SHOP/FIELD, ND, DESCRIPTION
    """
    items = []
    
    # Split into lines and process each line
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    for line in lines:
        # Pattern to match BOM format:
        # ID (number), QTY (number or number-number), SHOP/FIELD (word), ND (number"), DESCRIPTION
        pattern = r'(?P<id>\d+)\s*(?P<qty>[\d\'-]+)\s*(?P<location>SHOP|FIELD)\s*(?P<nd>[\d/]+\"?)\s*(?P<description>.*)'
        match = re.search(pattern, line)
        
        if match:
            item = {
                'id': match.group('id'),
                'qty': match.group('qty'),
                'location': match.group('location'),
                'nd': match.group('nd'),
                'description': match.group('description').strip()
            }
            items.append(item)
    
    return items

def process_pdf(pdf_path):
    """
    Process PDF and extract BOM information
    """
    doc = fitz.open(pdf_path)
    all_items = []
    
    for page_num in range(len(doc)):
        print(f"\nProcessing page {page_num + 1}...")
        page = doc[page_num]
        image_list = page.get_images()
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))
            
            # Preprocess and OCR
            processed_image = preprocess_image(image)
            # Use PSM 6 for uniform text block
            custom_config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(processed_image, config=custom_config)
            
            # Extract BOM data
            items = extract_bom_data(text)
            all_items.extend(items)
    
    doc.close()
    return all_items

def display_results(items):
    """
    Display the BOM results in a table format
    """
    print("\nBill of Materials:")
    print("=" * 100)
    print(f"{'ID':<5}{'QTY':<10}{'SHOP/FIELD':<12}{'ND':<8}{'DESCRIPTION':<65}")
    print("-" * 100)
    
    for item in items:
        print(f"{item['id']:<5}{item['qty']:<10}{item['location']:<12}{item['nd']:<8}{item['description']:<65}")

def main():
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Takeoff Drawing Example 2.pdf"
    try:
        # Process PDF and get structured data
        items = process_pdf(pdf_path)
        
        # Display results
        display_results(items)
        
    except Exception as e:
        print(f"Error processing PDF: {e}")

if __name__ == "__main__":
    main()


Processing page 1...

Processing page 2...

Processing page 3...

Bill of Materials:
ID   QTY       SHOP/FIELD  ND      DESCRIPTION                                                      
----------------------------------------------------------------------------------------------------


In [17]:
import fitz
import pytesseract
from PIL import Image
import io
import cv2
import numpy as np
import re

def preprocess_image(image):
    """
    Preprocess the image to improve OCR accuracy
    """
    # Convert PIL image to OpenCV format
    opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
    # Convert to grayscale
    gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
    
    # Apply thresholding
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    return Image.fromarray(thresh)

def crop_bom_section(image):
    """
    Crop the image to only include the Bill of Materials table
    Assuming the table is in the upper right quarter of the image
    """
    width, height = image.size
    # Crop to the upper right quarter (adjust these values based on your PDFs)
    right_portion = width * 0.6  # Start at 60% from the left
    bottom_portion = height * 0.4  # Take top 40% of the page
    
    cropped = image.crop((right_portion, 0, width, bottom_portion))
    return cropped

def extract_bom_data(text):
    """
    Extract Bill of Materials data with specific columns:
    ID, QTY, SHOP/FIELD, ND, DESCRIPTION
    """
    items = []
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    # Pattern to match BOM entries
    pattern = r'(?P<id>\d+)\s*(?P<qty>[\d\'-]+)\s*(?P<location>SHOP|FIELD)\s*(?P<nd>[\d/]+\"?)\s*(?P<description>.*)'
    
    for line in lines:
        match = re.search(pattern, line)
        if match:
            item = {
                'id': match.group('id'),
                'qty': match.group('qty'),
                'location': match.group('location'),
                'nd': match.group('nd'),
                'description': match.group('description').strip()
            }
            items.append(item)
    
    return items

def process_pdf(pdf_path):
    """
    Process PDF and extract BOM information
    """
    doc = fitz.open(pdf_path)
    all_items = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        image_list = page.get_images()
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            image = Image.open(io.BytesIO(image_bytes))
            
            # Crop to BOM section
            cropped_image = crop_bom_section(image)
            
            # Preprocess and OCR
            processed_image = preprocess_image(cropped_image)
            
            # Save processed image for verification (optional)
            processed_image.save(f'processed_bom_page_{page_num + 1}.png')
            
            # Perform OCR with custom configuration
            custom_config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(processed_image, config=custom_config)
            
            # Extract BOM data
            items = extract_bom_data(text)
            all_items.extend(items)
    
    doc.close()
    return all_items

def display_results(items):
    """
    Display the BOM results in a table format
    """
    print("\nBill of Materials:")
    print("=" * 100)
    print(f"{'ID':<5}{'QTY':<10}{'SHOP/FIELD':<12}{'ND':<8}{'DESCRIPTION':<65}")
    print("-" * 100)
    
    for item in items:
        print(f"{item['id']:<5}{item['qty']:<10}{item['location']:<12}{item['nd']:<8}{item['description']:<65}")

def main():
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Takeoff Drawing Example 2.pdf"
    try:
        # Process PDF and get structured data
        items = process_pdf(pdf_path)
        
        # Display results
        display_results(items)
        
    except Exception as e:
        print(f"Error processing PDF: {e}")

if __name__ == "__main__":
    main()


Bill of Materials:
ID   QTY       SHOP/FIELD  ND      DESCRIPTION                                                      
----------------------------------------------------------------------------------------------------
